# Health Policy RAG Assistant

Ask questions about 5 health-plan PDFs and get answers backed by **Azure OpenAI** and **ChromaDB**.

Flow: extract PDF text → chunk it → embed it → index in Chroma → retrieve → generate a cited answer.

Needs a `.env` file next to this notebook with your Azure OpenAI endpoint, key, and deployment names.

## Step 1 — Install libraries

In [1]:
%pip install -q openai chromadb pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Step 2 — Configuration

In [2]:
import glob
import os
import re

from dotenv import load_dotenv
from openai import AzureOpenAI
import chromadb
from pypdf import PdfReader

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
gen_deployment = os.getenv("AZURE_OPENAI_MODEL", "gpt-4.1")
embed_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
api_ver = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")

if not endpoint or not api_key:
    raise ValueError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

# Where the 5 policy PDFs live
PDF_PATH = os.path.join(os.getcwd(), "data", "healthcare_policies")
COLLECTION = "policies_2026"
RESULTS_K = 4

aoai_client = AzureOpenAI(azure_endpoint=endpoint, api_key=api_key, api_version=api_ver)
print("Azure OpenAI client is set up.")

Azure OpenAI client is set up.


## Step 3 — Extract PDF text

In [3]:
def load_documents(folder_path):
    files = sorted(glob.glob(os.path.join(folder_path, "*.pdf")))
    if not files:
        raise FileNotFoundError(f"No PDF files found in {folder_path!r}.")
    loaded = []
    for file_path in files:
        reader = PdfReader(file_path)
        text_content = "\n".join(page.extract_text() or "" for page in reader.pages)
        loaded.append({"source": os.path.basename(file_path), "text": text_content})
    return loaded

pdf_docs = load_documents(PDF_PATH)
print(f"Loaded {len(pdf_docs)} PDF file(s):")
for entry in pdf_docs:
    print(f"  - {entry['source']} ({len(entry['text'])} characters)")

Loaded 5 PDF file(s):
  - 01_Gold_PPO_2026_Benefits_Authorization.pdf (1919 characters)
  - 02_Silver_HMO_2026_Benefits_Authorization.pdf (1561 characters)
  - 03_Advanced_Imaging_Utilization_Management_2026.pdf (1713 characters)
  - 04_Rehabilitation_Therapy_Policy_2026.pdf (1329 characters)
  - 05_Claims_Benefits_Appeals_Operations_2026.pdf (1697 characters)


## Step 4 — Chunk the text

In [4]:
HEADER_REGEX = re.compile(r"(SECTION\s+\d+:[^\n]*)", re.IGNORECASE)

def make_chunks(doc_entry, size_limit=1200):
    parts = HEADER_REGEX.split(doc_entry["text"])
    output = []

    header_text = parts[0].strip()
    if header_text:
        output.append({"source": doc_entry["source"], "section": "Header/Metadata", "text": header_text})

    for i in range(1, len(parts), 2):
        section_title = parts[i].strip()
        section_body = parts[i + 1].strip() if i + 1 < len(parts) else ""
        merged = f"{section_title}\n{section_body}".strip()
        for offset in range(0, len(merged), size_limit):
            output.append({"source": doc_entry["source"], "section": section_title, "text": merged[offset:offset + size_limit]})

    return output

all_chunks = []
for entry in pdf_docs:
    all_chunks.extend(make_chunks(entry))
print(f"Total chunks generated: {len(all_chunks)}")

Total chunks generated: 34


## Step 5 — Embed and index in ChromaDB

Azure generates the embeddings; Chroma keeps them in memory and handles the similarity search.

In [5]:
def embed(text_batch, batch_size=64):
    output_vectors = []
    for i in range(0, len(text_batch), batch_size):
        piece = text_batch[i:i + batch_size]
        result = aoai_client.embeddings.create(model=embed_deployment, input=piece)
        output_vectors.extend(item.embedding for item in result.data)
    return output_vectors

vector_store = chromadb.Client()  # in-memory
try:
    vector_store.delete_collection(COLLECTION)
except Exception:
    pass
index = vector_store.create_collection(name=COLLECTION, metadata={"hnsw:space": "cosine"})

texts_for_index = [c["text"] for c in all_chunks]
vectors_for_index = embed(texts_for_index)

index.add(
    ids=[f"chunk_{n}" for n in range(len(all_chunks))],
    embeddings=vectors_for_index,
    documents=texts_for_index,
    metadatas=[{"source": c["source"], "section": c["section"]} for c in all_chunks],
)
print(f"Indexed {index.count()} chunks under '{COLLECTION}'.")

Indexed 34 chunks under 'policies_2026'.


## Step 6 — Retrieve relevant chunks

In [6]:
def find_relevant_chunks(question, k=RESULTS_K):
    question_vector = embed([question])
    matches = index.query(query_embeddings=question_vector, n_results=k)
    results = []
    for text, meta, distance in zip(matches["documents"][0], matches["metadatas"][0], matches["distances"][0]):
        results.append({
            "text": text,
            "source": meta["source"],
            "section": meta["section"],
            "score": 1 - distance,  # smaller distance = closer match
        })
    return results

for match in find_relevant_chunks("How many physical therapy visits before prior authorization on Gold PPO?"):
    print(f"[{match['score']:.3f}] {match['source']} | {match['section']}")

[0.769] 04_Rehabilitation_Therapy_Policy_2026.pdf | SECTION 2: PLAN-SPECIFIC PHYSICAL THERAPY THRESHOLDS
[0.672] 01_Gold_PPO_2026_Benefits_Authorization.pdf | SECTION 3: PHYSICAL THERAPY
[0.663] 02_Silver_HMO_2026_Benefits_Authorization.pdf | SECTION 3: PHYSICAL THERAPY
[0.588] 01_Gold_PPO_2026_Benefits_Authorization.pdf | SECTION 4: SPECIALIST SERVICES


## Step 7 — Generate a cited answer

In [7]:
SYSTEM_MESSAGE = (
    "You are a health-plan policy assistant. Only use the given context to answer. "
    "If the context doesn't cover it, say you don't have that information. "
    "Cite the source document and section for every fact you use."
)

def build_prompt(matches):
    entries = [f"[{i}] Source: {m['source']} | {m['section']}\n{m['text']}" for i, m in enumerate(matches, 1)]
    return "\n\n".join(entries)

def answer_question(question, k=RESULTS_K):
    matches = find_relevant_chunks(question, k)
    completion = aoai_client.chat.completions.create(
        model=gen_deployment,
        max_tokens=1000,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": f"Context:\n{build_prompt(matches)}\n\nQuestion: {question}"},
        ],
    )
    return completion.choices[0].message.content, matches

reply, used_chunks = answer_question(
    "How many physical therapy visits are allowed before prior authorization on Gold PPO vs Silver HMO?"
)
print(reply)
print("\n--- Sources ---")
for chunk in used_chunks:
    print(f"  [{chunk['score']:.3f}] {chunk['source']} | {chunk['section']}")

For Gold PPO, the first 10 physical therapy visits in a benefit year do not require prior authorization; prior authorization is required starting with the 11th visit (Source: 01_Gold_PPO_2026_Benefits_Authorization.pdf, SECTION 3: PHYSICAL THERAPY).

For Silver HMO, the first 6 physical therapy visits in a benefit year do not require prior authorization; prior authorization is required beginning with the 7th visit (Source: 02_Silver_HMO_2026_Benefits_Authorization.pdf, SECTION 3: PHYSICAL THERAPY).

--- Sources ---
  [0.791] 04_Rehabilitation_Therapy_Policy_2026.pdf | SECTION 2: PLAN-SPECIFIC PHYSICAL THERAPY THRESHOLDS
  [0.639] 01_Gold_PPO_2026_Benefits_Authorization.pdf | SECTION 3: PHYSICAL THERAPY
  [0.629] 02_Silver_HMO_2026_Benefits_Authorization.pdf | SECTION 3: PHYSICAL THERAPY
  [0.592] 01_Gold_PPO_2026_Benefits_Authorization.pdf | SECTION 4: SPECIALIST SERVICES


## Step 8 — Ask more questions

In [8]:
sample_questions = [
    "How long does the provider have to file an appeal after a denial or rejection?",
    "What is the prior authorization process?"
]

for q in sample_questions:
    reply, _ = answer_question(q)
    print(f"Q: {q}\nA: {reply}\n{'='*80}")

Q: How long does the provider have to file an appeal after a denial or rejection?
A: A provider should submit an appeal within 60 calendar days from the date of the denial notice unless a contract or applicable regulation specifies a different timeframe. The appeal should include the claim identifier, reason for dispute, and supporting documentation (Source: 05_Claims_Benefits_Appeals_Operations_2026.pdf, SECTION 5: PROVIDER APPEALS).


Q: What is the prior authorization process?
A: The prior authorization process involves obtaining approval from the health plan before proceeding with certain services to confirm that medical-necessity review requirements have been met. Specifically, for services that require prior authorization (such as advanced outpatient diagnostic imaging or physical therapy visits beyond a set threshold), the request must be made and approved before the scheduled service date. For physical therapy under the Gold PPO plan, prior authorization is required starting with the 11th visit, and the request must include the diagnosis, functional goals, progress to date, and proposed treatment frequency. It is important to note that prior authorization does not guarantee payment; final payment is still subject to eligibility, benefit limits, coding accuracy, and other plan provisions ([01_Gold_PPO_2026_Benefits_Authorization.pdf, Section 6]; [03_Advanced_Imaging_Utilization_Management_2026.pdf, Section 2]; 